# Capítulo 2 — ETL: Carga, Limpeza e Rotulagem
Exploramos os dados brutos, aplicamos normalização de CNPJ, datas e valores,
e geramos os datasets rotulados para os cenários de conciliação exata e fuzzy.

In [1]:
from reconciliacao.utils.config import load_config
from reconciliacao.etl.loader import load_pagamentos, load_nfse
from reconciliacao.etl.cleaner import clean_pagamentos, clean_nfse
from reconciliacao.etl.labeler import label_records
import pandas as pd

cfg = load_config("../config.yaml")
df_pag_raw = load_pagamentos("../data/raw/pagamentos.xlsx")
df_nfse_raw = load_nfse("../data/raw/nfse.xml")

In [2]:
print("=== Pagamentos Brutos ===")
print(df_pag_raw.dtypes)
print(df_pag_raw.head(5).to_string())

=== Pagamentos Brutos ===
id_pagamento       str
cnpj_fornecedor    str
data_pagamento     str
valor_pago         str
descricao          str
centro_custo       str
dtype: object
  id_pagamento cnpj_fornecedor       data_pagamento valor_pago                            descricao centro_custo
0   PAG-000001  39660734467499  2026-02-17 00:00:00   18228.97             embrace proactive niches       CC-372
1   PAG-000002  46316460664987  2025-09-17 00:00:00   19451.62             enhance global platforms       CC-896
2   PAG-000003  65923002242290  2026-03-03 00:00:00   34954.64       extend e-business applications       CC-819
3   PAG-000004  42597657125419  2025-11-24 00:00:00   49828.67  integrate proactive infrastructures       CC-204
4   PAG-000005  39064847739286  2025-09-06 00:00:00   16377.06            enhance proactive schemas       CC-237


In [3]:
df_pag = clean_pagamentos(df_pag_raw)
df_nfse = clean_nfse(df_nfse_raw)

print(f"CNPJs inválidos (pagamentos): {(~df_pag['cnpj_valid']).sum()}")
print(f"CNPJs inválidos (NFS-e): {(~df_nfse['nfse_cnpj_valid']).sum()}")
print(f"Duplicatas (pagamentos): {df_pag['is_duplicate'].sum()}")

CNPJs inválidos (pagamentos): 0
CNPJs inválidos (NFS-e): 543
Duplicatas (pagamentos): 0


In [4]:
for scenario in ["exact", "fuzzy"]:
    scfg = cfg["scenarios"][scenario]
    df = label_records(df_pag, df_nfse,
                       date_tolerance_days=scfg["date_tolerance_days"],
                       value_tolerance_pct=scfg["value_tolerance_pct"],
                       n_classes=scfg["n_classes"])
    df["scenario"] = scenario
    df.to_csv(f"../data/processed/{scenario}_reconciliado.csv", index=False)
    print(f"\n=== {scenario.upper()} ===")
    print(df["label"].value_counts().sort_index())


=== EXACT ===
label
0    2268
1    5250
Name: count, dtype: int64



=== FUZZY ===
label
0    1139
1    1129
2    5250
Name: count, dtype: int64
